In [9]:
import os 
from typing import Annotated, TypedDict 
from langchain_core.messages import BaseMessage, HumanMessage 
from langchain_groq import ChatGroq 
from langgraph.graph import END, START, StateGraph 
from langgraph.graph.message import add_messages 
from langgraph.checkpoint.memory import MemorySaver
import dotenv
dotenv.load_dotenv()

True

In [2]:
class ChatState(TypedDict): 
   # 'add_messages' ensures state appends messages rather than replacing them 
   messages: Annotated[list[BaseMessage], add_messages]

In [10]:
# load api key from .env file
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=api_key)

In [4]:
def chat_node(state: ChatState): 
   # Extract current conversation messages 
   messages = state["messages"] 
    
   # Invoke Grok API model 
   response = llm.invoke(messages) 
    
   # Return response in a list to merge via add_messages reducer 
   return {"messages": [response]}

In [5]:
# Create graph instance 
graph = StateGraph(ChatState) 
 
# Add chat node 
graph.add_node("chat_node", chat_node) 
 
# Set edges 
graph.add_edge(START, "chat_node") 
graph.add_edge("chat_node", END) 
 
# Add Memory Saver Checkpointer for conversation persistence 
checkpointer = MemorySaver() 
 
# Compile the graph with checkpointer 
chatbot = graph.compile(checkpointer=checkpointer) 


In [11]:
# Configure thread ID for the session 
config = {"configurable": {"thread_id": "1"}} 
 
print("Chatbot Initialized! Type 'exit', 'quit', or 'by' to stop.\n") 
 
while True: 
   user_input = input("You: ") 
    
   if user_input.strip().lower() in ["exit", "quit", "by"]: 
       print("Bot: Goodbye!") 
       break 
        
   # Send user input into state graph 
   initial_state = {"messages": [HumanMessage(content=user_input)]} 
    
   # Pass config containing thread_id to preserve state in RAM 
   result = chatbot.invoke(initial_state, config=config) 
    
   # Extract latest AI response 
   latest_response = result["messages"][-1].content 
   print(f"Bot: {latest_response}\n")

Chatbot Initialized! Type 'exit', 'quit', or 'by' to stop.

Bot: Hello! How can I help you today?

Bot: Your name is Qasim.

Bot: You’re welcome, Qasim! If there’s anything else you’d like to work on or ask, just let me know. Happy bot building!

Bot: Goodbye!
